# Benchmark 1: 2 Class vs 4 Class: Cross-Session

In [9]:
import numpy as np
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery, LeftRightImagery
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from mne.decoding import CSP
from moabb.evaluations import CrossSubjectEvaluation
from sklearn.pipeline import make_pipeline
from scipy import signal
from scipy.io import loadmat
import os
import mne
from scipy.linalg import logm, expm
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score
from pyriemann.estimation import Covariances
import scipy

In [1]:
from pyriemann.utils import mean_riemann
from scipy.optimize import minimize
from pymanopt import Problem
from pymanopt.manifolds import SpecialOrthogonalGroup
from pymanopt.optimizers import SteepestDescent
from pymanopt import Problem
from functools import partial
from pymanopt.function import numpy as pymanopt_numpy
import autograd.numpy as anp  # Autograd's NumPy replacement
from autograd import grad
import pymanopt
import autograd.scipy.linalg as linalg
from pyriemann.classification import MDM

In [3]:
def logm_approx(A):
    I = anp.eye(A.shape[0])  # Identity matrix
    return A - I - 0.5 * (A - I) @ (A - I) 

def frobenius_norm(X):
    return anp.sqrt(anp.sum(X**2))


In [4]:
active_all_event_ids = {'769': 769, '770': 770, '771': 771, '772': 772}
active_lr_event_ids = {'769': 769, '770': 770}
unknown_event_id = {'783': 783} 

In [5]:
data_dir = '/home/vishwa/eeg_tl/Recreating papers/BCICIV_2a'

In [6]:
def encode_labels(labels_list):
    encoded_list = []
    for labels in labels_list:
        # Create mapping from original labels to 0,1
        unique_labels = np.unique(labels)
        label_map = {unique_labels[i]: i for i in range(len(unique_labels))}
        
        # Apply mapping
        encoded = np.array([label_map[label] for label in labels])
        encoded_list.append(encoded)
    return encoded_list

In [7]:
# Define a causal bandpass filter function using a Butterworth design.
def causal_bandpass_filter(data, lowcut=8, highcut=30, fs=250, order=5):
    nyq = 0.5 * fs
    # Normalize the cutoff frequencies (Matlab's fir1 expects normalized cutoff frequencies
    low = lowcut / nyq
    high = highcut / nyq
    # Design the FIR filter. Note: order+1 coefficients are returned to match Matlab's fir1 which returns n+1 taps.
    b = signal.firwin(order + 1, [low, high], window='hamming', pass_zero=False)
    # Apply the filter causally using lfilter (this introduces a constant delay).
    filtered_data = signal.lfilter(b, [1.0], data)
    return filtered_data

In [ ]:
# With a sampling frequency of 250 Hz, 1001 samples equate to 1001/250 seconds.
sfreq = 250
tmin = 0.5       # Epoch start at cue onset.
# Set tmax so that n_samples = (tmax-tmin)*sfreq + 1 = 1001, i.e. 4 seconds long.
tmax = 3.5  # This gives 4.0 seconds.
filter_order

In [ ]:
def twofour_crosssession(n_classes):

    if(n_classes==2):
        event_ids = active_lr_event_ids
    else:
        event_ids = active_all_event_ids
    

    train_active_X = []         # List to hold numpy arrays with shape (n_trials, 22, 1001) per subject.
    train_active_y = []         # List to hold event labels per subject.
    train_active_metadata = []  # List to hold event metadata per subject.

    # Loop over subjects. Assume files are named "A01T.gdf", "A02T.gdf", ..., "A09T.gdf".
    for subj in range(1, 10):
        filename = os.path.join(data_dir, f'A{subj:02d}T.gdf')
        
        # Read the GDF file (using preload=True to load data into memory).
        train_raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False, eog=['EOG-left', 'EOG-central', 'EOG-right'])
        # Retain only EEG channels (22 channels) and exclude EOG channels.
        train_raw.pick_types(eeg=True, eog=False)
        
        # Extract events corresponding only to the four desired types.
        train_active_events, _ = mne.events_from_annotations(train_raw, event_id=event_ids)
        
        # Create epochs from tmin to tmax.
        train_active_epochs = mne.Epochs(train_raw, train_active_events, event_id=event_ids, tmin=tmin, tmax=tmax,
                            baseline=None, preload=True, verbose=False)
        
        # Get the epoch data (num_epochs x 22 channels x 1001 samples).
        train_active_data = train_active_epochs.get_data()
        
        # Print the number of extracted epochs to verify
        print(f"Subject {subj}: Epoch data shape {train_active_data.shape}")
        
        # Sampling frequency from raw.info (should be 250).
        fs = int(train_raw.info['sfreq'])
        # print(fs)
        n_trials, n_channels, n_times = train_active_data.shape
        train_active_filtered_data = np.empty_like(train_active_data)
        
        # Apply the causal bandpass filter channel‐wise for each trial.
        for trial in range(n_trials):
            for ch in range(n_channels):
                train_active_filtered_data[trial, ch, :] = causal_bandpass_filter(
                    train_active_data[trial, ch, :],
                    lowcut=8,    # Lower bound of sensorimotor rhythm.
                    highcut=30,  # Upper bound of sensorimotor rhythm.
                    fs=fs,
                    order=filter_order     # Lower order for a smoother causal filter.
                )
        # Apply the causal bandpass filter channel‐wise for each trial.
        # for trial in range(n_trials):
        #     for ch in range(n_channels):
        #         train_active_filtered_data[trial, ch, :] = train_active_data[trial, ch, :]
        
        # Append the processed data, labels, and event metadata.
        train_active_X.append(train_active_filtered_data)
        train_active_y.append(train_active_epochs.events[:, 2])  # The third column holds the event code.
        train_active_metadata.append(train_active_epochs.events)

    print("Loaded data for", len(train_active_X), "subjects.")

    eval_active_X = []         
    eval_active_y = []        
    eval_active_metadata = [] 

    # Loop over subjects. Assume files are named "A01T.gdf", "A02T.gdf", ..., "A09T.gdf".
    for subj in range(1, 10):
        filename = os.path.join(data_dir, f'A{subj:02d}E.gdf')
        mat_data = loadmat(f'/home/vishwa/eeg_tl/Recreating papers/BCICIV_2A true labels/A{subj:02d}E.mat')
        true_y =  np.array(mat_data['classlabel'], dtype=np.int64).reshape(288,) + 768
        # Read the GDF file (using preload=True to load data into memory).
        eval_raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False, eog=['EOG-left', 'EOG-central', 'EOG-right'])
        # Retain only EEG channels (22 channels) and exclude EOG channels.
        eval_raw.pick_types(eeg=True, eog=False)
        
        # Extract events corresponding only to the four desired types.
        eval_active_events, _ = mne.events_from_annotations(eval_raw, event_id=unknown_event_id)
        
        # Create epochs from tmin to tmax.
        eval_active_epochs = mne.Epochs(eval_raw, eval_active_events, event_id=unknown_event_id, tmin=tmin, tmax=tmax,
                            baseline=None, preload=True, verbose=False)
        
        # Get the epoch data (num_epochs x 22 channels x 1001 samples).
        eval_active_data = eval_active_epochs.get_data()
        
        # Print the number of extracted epochs to verify
        print(f"Subject {subj}: Epoch data shape {eval_active_data.shape}")
        
        # Sampling frequency from raw.info (should be 250).
        fs = int(eval_raw.info['sfreq'])
        # print(fs)
        n_trials, n_channels, n_times = eval_active_data.shape
        eval_active_filtered_data = np.empty_like(eval_active_data)
        
        # Apply the causal bandpass filter channel‐wise for each trial.
        for trial in range(n_trials):
            for ch in range(n_channels):
                eval_active_filtered_data[trial, ch, :] = causal_bandpass_filter(
                    eval_active_data[trial, ch, :],
                    lowcut=8,    # Lower bound of sensorimotor rhythm.
                    highcut=30,  # Upper bound of sensorimotor rhythm.
                    fs=fs,
                    order=filter_order     # Lower order for a smoother causal filter.
                )
        
        # for trial in range(n_trials):
        #     for ch in range(n_channels):
        #         eval_active_filtered_data[trial, ch, :] = eval_active_data[trial, ch, :]
                    
        
        # Append the processed data, labels, and event metadata.
        eval_active_X.append(eval_active_filtered_data)
        eval_active_y.append(true_y)  # The third column holds the event code.
        eval_active_metadata.append(eval_active_epochs.events)

    print("Loaded data for", len(eval_active_X), "subjects.")

    if(n_classes==2):
        eval_active_X = [x[np.isin(y, [769, 770])] for x, y in zip(eval_active_X, eval_active_y)]
        eval_active_y = [y[np.isin(y, [769, 770])] for y in eval_active_y]
    
    train_active_y = encode_labels(train_active_y)
    eval_active_y = encode_labels(eval_active_y)

    align_per_class = 14
    n_subjects = 9
    if(n_classes==2):
        classes = [0, 1]
    else:
        classes = [0, 1, 2, 3]  # Two classes as per your setup
    epsilon = 1e-6
    n_channels = 22
    accuracies = []

    for subj_idx in range(len(train_active_X)):
        # print(subj_idx)
        # Split data into train/test using leave-one-subject-out
        X_target = eval_active_X[subj_idx]
        y_target = eval_active_y[subj_idx]
        
        # Concatenate data from other subjects
        X_source = train_active_X[subj_idx]
        y_source = train_active_y[subj_idx]

        cov_estimator = Covariances(estimator='scm') 
        X_source = cov_estimator.fit_transform(X_source) 
        X_target = cov_estimator.fit_transform(X_target) 

        # Split target data into alignment and test sets
        align_indices = []
        test_indices = []
        for cls in classes:
            cls_indices = np.where(y_target == cls)[0]
            np.random.shuffle(cls_indices)
            align_indices.extend(cls_indices[:align_per_class])
            test_indices.extend(cls_indices[align_per_class:])

        M_source = mean_riemann(X_source)
        M_target_align = mean_riemann(X_target[align_indices])

        M_source_inv_half = np.linalg.inv(scipy.linalg.sqrtm(M_source))
        source_rct = [M_source_inv_half @ C @ M_source_inv_half for C in X_source]
        
        M_target_inv_half = np.linalg.inv(scipy.linalg.sqrtm(M_target_align))
        target_rct = [M_target_inv_half @ C @ M_target_inv_half for C in X_target]
        

        # Compute dispersion d for source
        print("Calculating source dispersions")
        d = 0
        for C_ret in source_rct:
            A_inv_half = np.linalg.inv(scipy.linalg.sqrtm(np.eye(n_channels)))
            C = np.dot(np.dot(A_inv_half, C_ret), A_inv_half)
            log_C = scipy.linalg.logm(C)
            d += np.linalg.norm(log_C, 'fro')**2

        # Compute dispersion tilde_d for T_l
        print("Calculating target dispersions")
        target_align_rct = [target_rct[i] for i in align_indices]
        tilde_d = 0
        for C_ret in target_align_rct:
            A_inv_half = np.linalg.inv(scipy.linalg.sqrtm(np.eye(n_channels)))
            C = np.dot(np.dot(A_inv_half, C_ret), A_inv_half)
            log_C = scipy.linalg.logm(C)
            tilde_d += np.linalg.norm(log_C, 'fro')**2

        # Compute scaling factor s
        s = np.sqrt(d / tilde_d)

        # Stretch target matrices
        target_str = [scipy.linalg.fractional_matrix_power(C_ret, s) for C_ret in target_rct]

        # Compute class means for source
        print("Calculating source class means")
        M_k = []
        for cls in classes:
            source_cls_rct = np.stack([source_rct[i] for i in range(len(y_source)) if y_source[i] == cls])
            source_mean_cls = mean_riemann(source_cls_rct, tol=1e-6, maxiter=100)
            M_k.append(source_mean_cls)

        # Compute class means for labeled target
        print("Calculating labelled target class means")
        tilde_M_k = []
        for cls in classes:
            align_cls_str = np.stack([target_str[i] for i in range(len(y_target[align_indices])) if y_target[align_indices][i] == cls])
            align_mean_cls = mean_riemann(align_cls_str, tol=1e-6, maxiter=100)
            tilde_M_k.append(align_mean_cls)

        # Optimize for U (rotation matrix)
        # Parameterize U = exp(A) where A is skew-symmetric
        manifold = SpecialOrthogonalGroup(n_channels)

        # Define the cost function
        @pymanopt.function.autograd(manifold)
        def cost(U):
            total = 0.0
            for k in range(len(classes)):
                tilde_M_k_inv_sqrt = linalg.inv(linalg.sqrtm(tilde_M_k[k]))
                transformed = anp.dot(anp.dot(U, M_k[k]), U.T)
                arg_logm = anp.dot(anp.dot(tilde_M_k_inv_sqrt, transformed), tilde_M_k_inv_sqrt)
                log_term = logm_approx(arg_logm)
                total += frobenius_norm(log_term) ** 2
            return total

        # Create the optimization problem
        problem = Problem(manifold=manifold, cost=cost)

        # Choose a solver and run it
        solver = SteepestDescent()
        U_opt = solver.run(problem)

        # Rotate target matrices
        U_opt = np.array(U_opt.point)
        target_rot = [np.dot(np.dot(U_opt.T, C_str), U_opt) for C_str in target_str]


        X_train = np.concatenate((np.array(source_rct), np.array([target_rot[i] for i in align_indices])))
        y_train = np.concatenate((y_source, y_target[align_indices]))

        mdm = MDM(metric='riemann')
        mdm.fit(X_train, y_train)

        X_test = np.array([target_rot[i] for i in test_indices])
        y_pred = mdm.predict(X_test)

        # Compute and print accuracy
        accuracy = accuracy_score(y_target[test_indices], y_pred)
        accuracies.append(accuracy)
    
    for subj_idx in range(len(train_active_X)):
        print(f"Subject {subj_idx+1} Test Accuracy: {accuracies[subj_idx]:.2f}")

    print(f"\nMean Cross-Validation Accuracy: {np.mean(accuracies):.2f} ± {np.std(accuracies):.2f}")
    

        
    

In [11]:
twofour_crosssession(2)

/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770']
Subject 1: Epoch data shape (144, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770']
Subject 2: Epoch data shape (144, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 9: Epoch data shape (288, 22, 751)
Loaded data for 9 subjects.
Calculating source dispersions
Calculating target dispersions
Calculating source class means
Calculating labelled target class means
Optimizing...
Iteration    Cost                       Gradient norm     
---------    -----------------------    --------------    
   1         +5.3252471273860902e+00    2.78945909e-01    
   2         +5.0710548649409350e+00    2.47040996e-01    
   3         +4.7703186138493443e+00    2.90195788e-01    
   4         +4.7355514848378082e+00    3.73565882e-01    
   5         +4.6111570597843379e+00    3.00825591e-01    
   6         +4.5132032975532503e+00    2.65471957e-01    
   7         +4.4362892611674010e+00    1.47182828e-01    
   8         +4.4183869081030416e+00    1.67747737e-01    
   9         +4.3959315082682622e+00    1.24939464e-01    
  10         +4.3

/tmp/ipykernel_99886/744147049.py:175: RuntimeWarning: logm result may be inaccurate, approximate err = 2.578470373343845e-13
  log_C = scipy.linalg.logm(C)
/tmp/ipykernel_99886/744147049.py:175: RuntimeWarning: logm result may be inaccurate, approximate err = 2.5923509131462267e-13
  log_C = scipy.linalg.logm(C)


Calculating target dispersions
Calculating source class means
Calculating labelled target class means
Optimizing...
Iteration    Cost                       Gradient norm     
---------    -----------------------    --------------    
   1         +9.4181467942029364e+00    2.76907986e-01    
   2         +9.1554907108182526e+00    2.63913385e-01    
   3         +8.8045078459594137e+00    2.46710370e-01    
   4         +8.7616763706585150e+00    3.16117542e-01    
   5         +8.6246659867864004e+00    1.94646709e-01    
   6         +8.5976215254472592e+00    2.28732292e-01    
   7         +8.5246151742518101e+00    9.65830880e-02    
   8         +8.5017092075075187e+00    1.27391824e-01    
   9         +8.4883433078915260e+00    1.35921939e-01    
  10         +8.4613449177343192e+00    5.19113980e-02    
  11         +8.4600537229063395e+00    1.70575006e-01    
  12         +8.4550747751398756e+00    1.58406097e-01    
  13         +8.4384729864636618e+00    1.06462517e-01    

/tmp/ipykernel_99886/744147049.py:175: RuntimeWarning: logm result may be inaccurate, approximate err = 3.117197354897748e-13
  log_C = scipy.linalg.logm(C)


Calculating target dispersions
Calculating source class means
Calculating labelled target class means
Optimizing...
Iteration    Cost                       Gradient norm     
---------    -----------------------    --------------    
   1         +6.0141579460389440e+00    2.99261840e-01    
   2         +5.7361206449533615e+00    2.79509773e-01    
   3         +5.3674300727011719e+00    2.64662988e-01    
   4         +5.3084305941977217e+00    2.91510298e-01    
   5         +5.1683451711868065e+00    1.29725570e-01    
   6         +5.1161883961902523e+00    1.33527595e-01    
   7         +5.0950482173726908e+00    1.31326637e-01    
   8         +5.0802871878851450e+00    1.28298912e-01    
   9         +5.0623941092898939e+00    8.59882253e-02    
  10         +5.0584634723810238e+00    1.06486922e-01    
  11         +5.0473012574849463e+00    5.04747400e-02    
  12         +5.0466880025633927e+00    1.15966318e-01    
  13         +5.0443395582176933e+00    1.06164567e-01    

In [12]:
twofour_crosssession(4)

/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770', '771', '772']
Subject 1: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770', '771', '772']
Subject 2: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

Calculating target dispersions
Calculating source class means
Calculating labelled target class means
Optimizing...
Iteration    Cost                       Gradient norm     
---------    -----------------------    --------------    
   1         +2.0786739313663229e+01    4.97841621e-01    
   2         +2.0350337342026400e+01    4.17913289e-01    
   3         +1.9752968903550013e+01    4.33416259e-01    
   4         +1.9490566134856429e+01    4.88684126e-01    
   5         +1.9342567790831474e+01    4.98880364e-01    
   6         +1.9122677908543086e+01    2.94572715e-01    
   7         +1.9048098020372763e+01    2.66316590e-01    
   8         +1.8989201851425772e+01    2.09453694e-01    
   9         +1.8952433756048329e+01    2.31485774e-01    
  10         +1.8934279307215299e+01    2.84366787e-01    
  11         +1.8886141565670950e+01    1.21376676e-01    
  12         +1.8869765886644714e+01    2.91002193e-01    
  13         +1.8823941121026714e+01    1.24579622e-01    

/tmp/ipykernel_99886/744147049.py:175: RuntimeWarning: logm result may be inaccurate, approximate err = 3.1258559466905907e-13
  log_C = scipy.linalg.logm(C)


Calculating target dispersions
Calculating source class means
Calculating labelled target class means
Optimizing...
Iteration    Cost                       Gradient norm     
---------    -----------------------    --------------    
   1         +1.0386787288250421e+01    9.94542263e-01    
   2         +9.7227847906203237e+00    7.03348303e-01    
   3         +9.3114855925172364e+00    1.00150791e+00    
   4         +8.7765441227925312e+00    7.75181778e-01    
   5         +8.5954985073160959e+00    8.81918963e-01    
   6         +8.3118956182384149e+00    4.09355901e-01    
   7         +8.2472779975344910e+00    9.33590511e-01    
   8         +8.0553371321254854e+00    4.94642222e-01    
   9         +8.0082180166047916e+00    5.88815713e-01    
  10         +7.8966317934582619e+00    2.35096434e-01    
  11         +7.8278390731254222e+00    4.80847388e-01    
  12         +7.7729516301106250e+00    3.70412595e-01    
  13         +7.7236063716300496e+00    1.63569265e-01    

/tmp/ipykernel_99886/744147049.py:175: RuntimeWarning: logm result may be inaccurate, approximate err = 7.837678421674387e-13
  log_C = scipy.linalg.logm(C)


Calculating target dispersions
Calculating source class means
Calculating labelled target class means
Optimizing...
Iteration    Cost                       Gradient norm     
---------    -----------------------    --------------    
   1         +1.2362385379751650e+01    1.23749389e+00    
   2         +1.1578770792474611e+01    1.00316858e+00    
   3         +1.1210699922547281e+01    2.55211439e+00    
   4         +1.0425112430671476e+01    9.20515876e-01    
   5         +1.0419193004096121e+01    1.40609176e+00    
   6         +1.0395640232045373e+01    1.39158580e+00    
   7         +1.0303452449157895e+01    1.33461056e+00    
   8         +9.9698689135257883e+00    1.10626923e+00    
   9         +9.3820434756918107e+00    7.15537713e-01    
  10         +9.1988382586374495e+00    7.69721133e-01    
  11         +9.1561548980902234e+00    9.62463605e-01    
  12         +9.0085583706243657e+00    7.03666400e-01    
  13         +8.9997189905986907e+00    9.28091580e-01    